# StyleTTS 2 Demo (SPT-CRo)

In [ ]:
%cd ..

### Randomness

In [ ]:
import torch

torch.manual_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

import random

random.seed(0)

import numpy as np

np.random.seed(0)

### Imports and packages

In [ ]:
# load packages
import sys
import os
import time
import yaml
import torch
import torch.nn.functional as F
import torchaudio
import librosa
# from nltk.tokenize import word_tokenize

from models import load_ASR_models, load_F0_models, build_model
from Modules.diffusion.sampler import (ADPM2Sampler, DiffusionSampler,
                                       KarrasSchedule)
from Utils.PLBERT.util import load_plbert
from utils import recursive_munch
from text_utils import TextCleaner

from tpp_ttstool import TppTtstool

%matplotlib inline
import IPython.display as ipd

### Functions and definitions

In [ ]:
to_mel = torchaudio.transforms.MelSpectrogram(
    n_mels=80, n_fft=2048, win_length=1200, hop_length=300
)
mean, std = -4, 4


def length_to_mask(lengths):
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    mask = torch.gt(mask + 1, lengths.unsqueeze(1))
    return mask


def preprocess(wave):
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor = to_mel(wave_tensor)
    mel_tensor = (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std
    return mel_tensor


def compute_style(model, path):
    wave, sr = librosa.load(path, sr=24000)
    audio, index = librosa.effects.trim(wave, top_db=30)
    if sr != 24000:
        audio = librosa.resample(audio, sr, 24000)
    mel_tensor = preprocess(audio).to(DEVICE)

    with torch.no_grad():
        ref_s = model.style_encoder(mel_tensor.unsqueeze(1))
        ref_p = model.predictor_encoder(mel_tensor.unsqueeze(1))

    return torch.cat([ref_s, ref_p], dim=1)

In [ ]:
# Set username
USER = os.environ["USER"]

TPP_PATH = f"/storage/plzen4-ntis/home/{USER}/GIT_repos/TPP/src"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Set path to TPP
sys.path.insert(0, TPP_PATH)

# Define bin and data for phonemizer
TTSTOOL_BIN_PATH = "./Utils/tts_tool/tts_tool"
TTSTOOL_DATA_PATH = "./Utils/tts_tool/data/frontend_ph-redu.json"

### Load models

In [ ]:
# Set up TPP
tpp = TppTtstool("cs-cz", tts_tool_bin=TTSTOOL_BIN_PATH, tts_tool_data=TTSTOOL_DATA_PATH)

# Load processed training config
config = yaml.safe_load(open("Exps/SPT-CRo/config2.processed.yml"))

text_cleaner = TextCleaner(
    config["data_params"]["symbol_dict_path"], pad=config["data_params"]["pad"]
)
print(f"Number of symbols: {len(text_cleaner)}")
assert len(text_cleaner) == 81, f"Number of symbols must be 81 but it is {len(text_cleaner)}"

# Load pretrained models
text_aligner = load_ASR_models(config["ASR_path"], config["ASR_config"])  # Text aligner
pitch_extractor = load_F0_models(config["F0_path"])  # F0 extractor
plbert = load_plbert(config["PLBERT_dir"])  # PLBERT

In [ ]:
# Build StyleTTS2 model
model_params = recursive_munch(config["model_params"])
model = build_model(model_params, text_aligner, pitch_extractor, plbert)
_ = [model[key].eval() for key in model]
_ = [model[key].to(DEVICE) for key in model]

In [ ]:
params = torch.load("Exps/SPT-CRo/model4tts.pth", map_location="cpu")

In [ ]:
for key in model:
    if key in params:
        print("%s loaded" % key)
        try:
            model[key].load_state_dict(params[key])
        except:
            from collections import OrderedDict

            state_dict = params[key]
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:]  # remove `module.`
                new_state_dict[name] = v
            # load params
            model[key].load_state_dict(new_state_dict, strict=False)
#             except:
#                 _load(params[key], model[key])
_ = [model[key].eval() for key in model]

In [ ]:
from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

In [ ]:
sampler = DiffusionSampler(
    model.diffusion.diffusion,
    sampler=ADPM2Sampler(),
    sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0),  # empirical parameters
    clamp=False,
)

In [ ]:
offset_beg = config["preprocess_params"]["silence_beg"]
offset_end = config["preprocess_params"]["silence_end"]

### Synthesize speech

In [ ]:
def synthesize(
    text,
    model,
    tpp,
    text_cleaner,
    ref_s,
    sampler,
    diffusion_steps=5,
    embedding_scale=1,
    alpha=0.3,
    beta=0.7,
    silence_beg=4800,
    silence_end=4800,
    device="cuda",
):
    # Clean text
    text = text.strip()
    text = text.replace('"', "")

    # Prepare phonemizer
    tpp.ssml_parse(text)

    # Initialize previous style and wavs
    wavs = []
    # s_prev = None  # reset style of previous sentence `s_prev`

    # Iterate over sentences
    for ps in tpp.to_sentences_phon():
        if not ps.strip():  # skip empty phonetic string
            continue

        tokens = [0] + text_cleaner(ps)  # add padding and tokenize phonetic string

        wav = inference(
            torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0),
            model,
            ref_s,
            sampler,
            diffusion_steps=diffusion_steps,
            embedding_scale=embedding_scale,
            alpha=alpha,
            beta=beta,
            device=device,
        )
        wavs.append(wav)

    return wavs

In [ ]:
def inference(
    tokens,
    model,
    ref_s,
    sampler,
    diffusion_steps=5,
    embedding_scale=1,
    alpha=0.3,
    beta=0.7,
    silence_beg=4800,
    silence_end=4800,
    device="cuda",
):
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2)

        s_pred = sampler(
            noise=torch.randn((1, 256)).unsqueeze(1).to(device),
            embedding=bert_dur,
            embedding_scale=embedding_scale,
            features=ref_s,  # reference from the same speaker as the embedding
            num_steps=diffusion_steps,
        ).squeeze(1)

        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        ref = alpha * ref + (1 - alpha) * ref_s[:, :128]
        s = beta * s + (1 - beta) * ref_s[:, 128:]

        d = model.prosodic_predictor.text_encoder(d_en, s, input_lengths, text_mask)

        x, _ = model.prosodic_predictor.lstm(d)
        duration = model.prosodic_predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame : c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device)
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.prosodic_predictor.F0Ntrain(en, s)

        asr = t_en @ pred_aln_trg.unsqueeze(0).to(device)
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, F0_pred, N_pred, ref.squeeze().unsqueeze(0))

    # return out.squeeze().cpu().numpy()[..., :-50] # weird pulse at the end of the model, need to be fixed later
    return out.squeeze().cpu().numpy()[silence_beg:-silence_end]

#### Basic synthesis (5 diffusion steps, seen speakers)

In [ ]:
# synthesize a text
text = """Šestašedesátiletý nadšený hráč stolního tenisu vlastní mnoho objektů v kraji."""

In [ ]:
reference_dicts = {}
reference_dicts["bosak"] = "Demo/reference_audios/SPT-CRo.cs/seen/bosak_jaromir.wav"
reference_dicts["holubova"] = "Demo/reference_audios/SPT-CRo.cs/seen/holubova_eva.wav"

`alpha` and `beta` are the factors that determine how much we use the style sampled based on the text instead of the reference. The higher the value of `alpha` and `beta`, the more suitable the style it is to the text but less similar to the reference. Using a higher `beta` makes the synthesized speech more emotional, at the cost of lower similarity to the reference. `alpha` determines the timbre of the speaker, while beta determines the prosody.

In [ ]:
# Default values of `alpha` (0.3) and `beta` (0.7) may result in noisy synthetic speech !!!
# => lower the values or switch them off (0.0)
alpha = 0
beta = 0

start = time.time()
# noise = torch.randn(1,1,256).to(DEVICE)
for k, path in reference_dicts.items():
    ref_s = compute_style(model, path)

    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=5,
        embedding_scale=1.0,
        alpha=alpha,
        beta=beta,
        device=DEVICE,
    )
    wav = np.concatenate(wavs)

    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    print(k + " Synthesized:")
    display(ipd.Audio(wav, rate=24000))
    print(k + " Reference:")
    display(ipd.Audio(path, rate=24000, normalize=False))

#### With higher diffusion steps (more diverse)

Since the sampler is ancestral, the higher the steps, the more diverse the samples are, with the cost of slower synthesis speed.

In [ ]:
alpha = 0
beta = 0

start = time.time()
# noise = torch.randn(1,1,256).to(DEVICE)
for k, path in reference_dicts.items():
    ref_s = compute_style(model, path)
    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=1.0,
        alpha=alpha,
        beta=beta,
        device=DEVICE,
    )
    wav = np.concatenate(wavs)

    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    print(k + " Synthesized:")
    display(ipd.Audio(wav, rate=24000))
    print(k + " Reference:")
    display(ipd.Audio(path, rate=24000, normalize=False))

#### Basic synthesis (5 diffusion steps, unseen speakers)
The following samples are to reproduce samples in [Section 4](https://styletts2.github.io/#libri) of the demo page. All spsakers are unseen during training. You can compare the generated samples to popular zero-shot TTS models like Vall-E and NaturalSpeech 2.

In [ ]:
reference_dicts = {}
# format: (path, text)
reference_dicts["babis"] = (
    "Demo/reference_audios/SPT-CRo.cs/unseen/babis_andrej.wav",
    "Akcie komerční banky, poměrně zřetelně oslabily, navzdory pozitivním zprávám, i příznivě, laděnému konsensu, mezi obchodníky.",
)
reference_dicts["drabova"] = (
    "Demo/reference_audios/SPT-CRo.cs/unseen/drabova_dana.wav",
    "Ministerstvo pro hospodářskou soutěž toto stanovisko zahraniční konkurence, sice nepotvrdilo, nicméně výsledky výběru anulovalo.",
)

In [ ]:
alpha = 0
beta = 0

# noise = torch.randn(1,1,256).to(DEVICE)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(model, path)
    start = time.time()
    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=1.0,
        alpha=alpha,
        beta=beta,
        device=DEVICE,
    )
    wav = np.concatenate(wavs)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    print(k + " Synthesized:")
    display(ipd.Audio(wav, rate=24000))
    print(k + " Reference:")
    display(ipd.Audio(path, rate=24000, normalize=False))

In [ ]:
reference_dicts = {}
reference_dicts["Babiš"] = (
    "/storage/plzen4-ntis/projects/public/Lehecka/TSD2024/wav/babis_andrej/listen_test_example1_orig.wav",
    "Tuhle větu jsem nikdy neřekl. Byla vygenerována pomocí umělé inteligence.",
)
reference_dicts["Drábová"] = (
    "/storage/plzen4-ntis/projects/public/Lehecka/TSD2024/wav/drabova_dana/listen_test_example1_orig.wav",
    "Tuhle větu jsem nikdy neřekl. Byla vygenerována pomocí umělé inteligence.",
)
reference_dicts["Matoušek"] = (
    "/storage/plzen4-ntis/projects/public/Lehecka/TSD2024/wav/matousek_jindrich/listen_test_example1_orig.wav",
    "Tuhle větu jsem nikdy neřekl. Byla vygenerována pomocí umělé inteligence.",
)
reference_dicts["Schwarzenberg"] = (
    "/storage/plzen4-ntis/projects/public/Lehecka/TSD2024/wav/schwarzenberg_karel/listen_test_example1_orig.wav",
    "Tuhle větu jsem nikdy neřekl. Byla vygenerována pomocí umělé inteligence.",
)

In [ ]:
# noise = torch.randn(1,1,256).to(DEVICE)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=1.0,
        alpha=0.3,
        beta=0.7,
        device=DEVICE,
    )
    wav = np.concatenate(wavs)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    print(k + " Synthesized:")
    display(ipd.Audio(wav, rate=24000))
    print(k + " Reference:")
    display(ipd.Audio(path, rate=24000, normalize=False))

### Speech expressiveness

The following section recreates the samples shown in [Section 6](https://styletts2.github.io/#emo) of the demo page. The speaker reference used is `1221-135767-0014.wav`, which is unseen during training. 

#### With `embedding_scale=1`
This is the classifier-free guidance scale. The higher the scale, the more conditional the style is to the input text and hence more emotional.



In [ ]:
ref_s = compute_style(
    "Data/SPT-MGW.cs/SPT-MGW/spkr00478/wavs/spkr00478_REC-SPT-NTB-2-1-13_000023.wav"
)

In [ ]:
texts = {}
texts["Happy"] = (
    "Jsme rádi, že vás můžeme pozvat na cestu do minulosti, kde navštívíme ty nejúžasnější památky, jaké kdy lidské ruce postavily."
)
texts["Sad"] = (
    "Mrzí mě, že musím říct, že jsme utrpěli vážnou ránu v našem úsilí obnovit prosperitu a důvěru."
)
texts["Angry"] = (
    "Obor astronomie je vtip! Jeho teorie jsou založeny na chybných pozorováních a zaujatých interpretacích!"
)
texts["Surprised"] = (
    "Nemohu tomu uvěřit! Chcete mi říct, že jste v tomto rybníku objevili nový druh bakterie?"
)

for k, v in texts.items():
    wavs = synthesize(
        v,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=1.0,
        alpha=0.3,
        beta=0.7,
        device=DEVICE,
    )
    wav = np.concatenate(wavs)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### With `embedding_scale=2`

In [ ]:
texts = {}
texts["Happy"] = (
    "Jsme rádi, že vás můžeme pozvat na cestu do minulosti, kde navštívíme ty nejúžasnější památky, jaké kdy lidské ruce postavily!"
)
texts["Sad"] = (
    "Mrzí mě, že musím říct, že jsme utrpěli vážnou ránu v našem úsilí obnovit prosperitu a důvěru."
)
texts["Angry"] = (
    "Obor astronomie je vtip! Jeho teorie jsou založeny na chybných pozorováních a zaujatých interpretacích!"
)
texts["Surprised"] = (
    "Nemohu tomu uvěřit! Chcete mi říct, že jste v tomto rybníku objevili nový druh bakterie?"
)

for k, v in texts.items():
    noise = torch.randn(1, 1, 256).to(DEVICE)
    wavs = synthesize(
        v,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=2.0,
        alpha=0.3,
        beta=0.7,
        device=DEVICE,
    )
    wav = np.concatenate(wavs)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### With `embedding_scale=2, alpha = 0.5, beta = 0.9`
`alpha` and `beta` is the factor to determine much we use the style sampled based on the text instead of the reference. The higher the value of `alpha` and `beta`, the more suitable the style it is to the text but less similar to the reference. Using higher beta makes the synthesized speech more emotional, at the cost of lower similarity to the reference. `alpha` determines the timbre of the speaker while `beta` determines the prosody. 

In [ ]:
texts = {}
texts["Happy"] = (
    "Jsme rádi, že vás můžeme pozvat na cestu do minulosti, kde navštívíme ty nejúžasnější památky, jaké kdy lidské ruce postavily!"
)
texts["Sad"] = (
    "Mrzí mě, že musím říct, že jsme utrpěli vážnou ránu v našem úsilí obnovit prosperitu a důvěru."
)
texts["Angry"] = (
    "Obor astronomie je vtip! Jeho teorie jsou založeny na chybných pozorováních a zaujatých interpretacích!"
)
texts["Surprised"] = (
    "Nemohu tomu uvěřit! Chcete mi říct, že jste v tomto rybníku objevili nový druh bakterie?"
)

for k, v in texts.items():
    noise = torch.randn(1, 1, 256).to(DEVICE)
    wavs = synthesize(
        v,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=2.0,
        alpha=0.3,
        beta=0.7,
        device=DEVICE,
    )
    wav = np.concatenate(wavs)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

### Zero-shot speaker adaptation
This section recreates the "Acoustic Environment Maintenance" and "Speaker’s Emotion Maintenance" demo in [Section 4](https://styletts2.github.io/#libri) of the demo page. You can compare the generated samples to popular zero-shot TTS models like Vall-E. Note that the model was trained only on LibriTTS, which is about 250 times fewer data compared to those used to trian Vall-E with similar or better effect for these maintainance. 

#### Acoustic Environment Maintenance

Since we want to maintain the acoustic environment in the speaker (timbre), we set  `alpha = 0` to make the speaker as closer to the reference as possible while only changing the prosody according to the text.  

In [ ]:
reference_dicts = {}
# format: (path, text)
reference_dicts["1"] = (
    "Data/SPT-MGW.cs/SPT-MGW/spkr00001/wavs/spkr00001_Speaker001_1_0000021.wav",
    "Co se týče přátel, rozhodně radši kamarádím s ženami.",
)
reference_dicts["478"] = (
    "Data/SPT-MGW.cs/SPT-MGW/spkr00478/wavs/spkr00478_REC-SPT-NTB-2-1-13_000023.wav",
    "Všechno řídí počítač, ale musíte umět přemýšlet, než můžete používat počítač.",
)
reference_dicts["122"] = (
    "Data/SPT-MGW.cs/SPT-MGW/spkr00122/wavs/spkr00122_Speaker122_1_0000021.wav",
    "A pak v Česku, tam máte v rámci Moravy úplně jinou situaci, se kterou si musíte dělat starosti.",
)

In [ ]:
noise = torch.randn(1, 1, 256).to(DEVICE)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=5,
        embedding_scale=1,
        alpha=0.0,
        beta=0.5,
        device=DEVICE,
    )
    wav = np.concatenate(wavs)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd

    print("Synthesized: " + text)
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print("Reference:")
    display(ipd.Audio(path, rate=24000, normalize=False))

#### Speaker’s Emotion Maintenance

Since we want to maintain the emotion in the speaker (prosody), we set  `beta = 0.1` to make the speaker as closer to the reference as possible while having some diversity thruogh the slight timbre change.

In [ ]:
reference_dicts = {}
## format: (path, text)
# reference_dicts['Anger'] = ("Demo/reference_audio/anger.wav", "We have to reduce the number of plastic bags.")
# reference_dicts['Sleepy'] = ("Demo/reference_audio/sleepy.wav", "We have to reduce the number of plastic bags.")
# reference_dicts['Amused'] = ("Demo/reference_audio/amused.wav", "We have to reduce the number of plastic bags.")
# reference_dicts['Disgusted'] = ("Demo/reference_audio/disgusted.wav", "We have to reduce the number of plastic bags.")

reference_dicts["1"] = (
    "Data/SPT-MGW.cs/SPT-MGW/spkr00001/wavs/spkr00001_Speaker001_1_0000021.wav",
    "Co se týče přátel, rozhodně radši kamarádím s ženami.",
)
reference_dicts["478"] = (
    "Data/SPT-MGW.cs/SPT-MGW/spkr00478/wavs/spkr00478_REC-SPT-NTB-2-1-13_000023.wav",
    "Všechno řídí počítač, ale musíte umět přemýšlet, než můžete používat počítač.",
)
reference_dicts["122"] = (
    "Data/SPT-MGW.cs/SPT-MGW/spkr00122/wavs/spkr00122_Speaker122_1_0000021.wav",
    "A pak v Česku, tam máte v rámci Moravy úplně jinou situaci, se kterou si musíte dělat starosti.",
)

In [ ]:
noise = torch.randn(1, 1, 256).to(DEVICE)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=1,
        alpha=0.3,
        beta=0.1,
        device=DEVICE,
    )
    wav = np.concatenate(wavs)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd

    print(k + " Synthesized: " + text)
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print(k + " Reference:")
    display(ipd.Audio(path, rate=24000, normalize=False))

### Longform Narration

This section includes basic implementation of Algorithm 1 in the paper for consistent longform audio generation. The example passage is taken from [Section 5](https://styletts2.github.io/#long) of the demo page.

In [ ]:
# passage = '''If the supply of fruit is greater than the family needs, it may be made a source of income by sending the fresh fruit to the market if there is one near enough, or by preserving, canning, and making jelly for sale. To make such an enterprise a success the fruit and work must be first class. There is magic in the word "Homemade," when the product appeals to the eye and the palate; but many careless and incompetent people have found to their sorrow that this word has not magic enough to float inferior goods on the market. As a rule large canning and preserving establishments are clean and have the best appliances, and they employ chemists and skilled labor. The home product must be very good to compete with the attractive goods that are sent out from such establishments. Yet for first class home made products there is a market in all large cities. All first-class grocers have customers who purchase such goods.'''
passage = """Pokud je zásoba ovoce větší, než rodina potřebuje, může se stát zdrojem příjmu buď prodejem čerstvého ovoce na trhu, pokud je dostatečně blízko, nebo jeho zpracováním, konzervováním a výrobou želé na prodej. Aby takový podnik byl úspěšný, musí být ovoce i práce prvotřídní. Slovo domácí výroba má své kouzlo, pokud produkt lahodí oku i chuti, ale mnoho nedbalých a nezkušených lidí zjistilo ke své lítosti, že toto slovo nemá dostatečné kouzlo k prosazení nekvalitního zboží na trhu. Obecně platí, že velké závody na konzervování a zpracování ovoce jsou čisté, mají nejlepší vybavení a zaměstnávají chemiky i kvalifikované pracovníky. Domácí výrobek musí být velmi kvalitní, aby mohl konkurovat atraktivnímu zboží z těchto podniků. Přesto existuje pro prvotřídní domácí produkty trh ve všech velkých městech. Všichni prvotřídní obchodníci s potravinami mají zákazníky, kteří takové zboží kupují."""

In [ ]:
def LFsynthesize(
    text,
    model,
    tpp,
    text_cleaner,
    ref_s,
    sampler,
    diffusion_steps=5,
    embedding_scale=1,
    alpha=0.3,
    beta=0.7,
    t=0.7,
    silence_beg=4800,
    silence_end=4800,
    device="cuda",
):
    # Clean text
    text = text.strip()
    text = text.replace('"', "")

    # Prepare phonemizer
    tpp.ssml_parse(text)

    # Initialize previous style and wavs
    wavs = []
    s_prev = None  # reset style of previous sentence `s_prev`

    # Iterate over sentences
    for ps in tpp.to_sentences_phon():
        if not ps.strip():  # skip empty phonetic string
            continue

        tokens = [0] + text_cleaner(ps)  # add padding and tokenize phonetic string

        wav = LFinference(
            torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0),
            model,
            s_prev,
            ref_s,
            sampler,
            diffusion_steps=diffusion_steps,
            embedding_scale=embedding_scale,
            alpha=alpha,
            beta=beta,
            t=t,
            device=device,
        )
        wavs.append(wav)

    return wavs

In [ ]:
def LFinference(
    tokens,
    model,
    s_prev,
    ref_s,
    sampler,
    diffusion_steps=5,
    embedding_scale=1,
    alpha=0.3,
    beta=0.7,
    t=0.7,
    silence_beg=4800,
    silence_end=4800,
    device="cuda",
):
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2)

        s_pred = sampler(
            noise=torch.randn((1, 256)).unsqueeze(1).to(device),
            embedding=bert_dur,
            embedding_scale=embedding_scale,
            features=ref_s,  # reference from the same speaker as the embedding
            num_steps=diffusion_steps,
        ).squeeze(1)

        if s_prev is not None:
            # convex combination of previous and current style
            s_pred = (1 - t) * s_prev + t * s_pred

        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        ref = alpha * ref + (1 - alpha) * ref_s[:, :128]
        s = beta * s + (1 - beta) * ref_s[:, 128:]

        s_pred = torch.cat([ref, s], dim=-1)

        d = model.prosodic_predictor.text_encoder(d_en, s, input_lengths, text_mask)

        x, _ = model.prosodic_predictor.lstm(d)
        duration = model.prosodic_predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame : c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device)
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.prosodic_predictor.F0Ntrain(en, s)

        asr = t_en @ pred_aln_trg.unsqueeze(0).to(device)
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, F0_pred, N_pred, ref.squeeze().unsqueeze(0))

    # return out.squeeze().cpu().numpy()[..., :-100], s_pred # weird pulse at the end of the model, need to be fixed later
    return out.squeeze().cpu().numpy()[silence_beg:-silence_end]

In [ ]:
# unseen speaker
path = "Data/SPT-MGW.cs/SPT-MGW/spkr00001/wavs/spkr00001_Speaker001_1_0000021.wav"
s_ref = compute_style(path)

wavs = LFsynthesize(
    passage,
    model,
    tpp,
    text_cleaner,
    s_ref,
    sampler,
    diffusion_steps=10,
    embedding_scale=1.0,
    alpha=0.3,
    beta=0.7,  # make it more suitable for the text
    t=0.7,
    silence_beg=4800,
    silence_end=4800,
    device=DEVICE,
)
wav = np.concatenate(wavs)

print("Synthesized: ")
display(ipd.Audio(np.concatenate(wavs), rate=24000, normalize=False))
print("Reference: ")
display(ipd.Audio(path, rate=24000, normalize=False))

### Style Transfer

The following section demostrates the style transfer capacity for unseen speakers in [Section 6](https://styletts2.github.io/#emo) of the demo page. For this, we set `alpha=0.5, beta = 0.9` for the most pronounced effects (mostly using the sampled style). 

In [ ]:
def STsynthesize(
    text,
    model,
    tpp,
    text_cleaner,
    ref_s,
    ref_text,
    ref_tpp,
    sampler,
    diffusion_steps=5,
    embedding_scale=1,
    alpha=0.3,
    beta=0.7,
    silence_beg=4800,
    silence_end=4800,
    device="cuda",
):
    # Clean text
    text = text.strip()
    text = text.replace('"', "")

    # Prepare phonemizer
    tpp.ssml_parse(text)

    # Prepare reference text
    ref_text = ref_text.strip()
    ref_text = ref_text.replace('"', "")
    ref_tpp.ssml_parse(ref_text)
    ref_tokens = []
    for ref_ps in ref_tpp.to_sentences_phon():
        ref_tokens.extend([0] + text_cleaner(ref_ps))

    # Initialize previous style and wavs
    wavs = []

    # Iterate over sentences
    for ps in tpp.to_sentences_phon():
        if not ps.strip():  # skip empty phonetic string
            continue

        tokens = [0] + text_cleaner(ps)  # add padding and tokenize phonetic string

        wav = STinference(
            torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0),
            model,
            ref_s,
            torch.tensor(ref_tokens, dtype=torch.long, device=device).unsqueeze(0),
            sampler,
            diffusion_steps=diffusion_steps,
            embedding_scale=embedding_scale,
            alpha=alpha,
            beta=beta,
            silence_beg=silence_beg,
            silence_end=silence_end,
            device=device,
        )
        wavs.append(wav)

    return wavs

In [ ]:
def STinference(
    tokens,
    model,
    ref_s,
    ref_tokens,
    sampler,
    diffusion_steps=5,
    embedding_scale=1,
    alpha=0.3,
    beta=0.7,
    silence_beg=4800,
    silence_end=4800,
    device="cuda",
):
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2)

        ref_input_lengths = torch.LongTensor([ref_tokens.shape[-1]]).to(device)
        ref_text_mask = length_to_mask(ref_input_lengths).to(device)
        ref_bert_dur = model.bert(ref_tokens, attention_mask=(~ref_text_mask).int())
        s_pred = sampler(
            noise=torch.randn((1, 256)).unsqueeze(1).to(device),
            embedding=bert_dur,
            embedding_scale=embedding_scale,
            features=ref_s,  # reference from the same speaker as the embedding
            num_steps=diffusion_steps,
        ).squeeze(1)

        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        ref = alpha * ref + (1 - alpha) * ref_s[:, :128]
        s = beta * s + (1 - beta) * ref_s[:, 128:]

        d = model.prosodic_predictor.text_encoder(d_en, s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame : c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device)
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = t_en @ pred_aln_trg.unsqueeze(0).to(device)
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, F0_pred, N_pred, ref.squeeze().unsqueeze(0))

    # return out.squeeze().cpu().numpy()[..., :-50] # weird pulse at the end of the model, need to be fixed later
    return out.squeeze().cpu().numpy()[silence_beg:-silence_end]

In [ ]:
# reference texts to sample styles

ref_texts = {}
ref_texts["Happy"] = (
    "Jsme rádi, že vás můžeme pozvat na cestu do minulosti, kde navštívíme ty nejúžasnější památky, jaké kdy lidské ruce postavily!"
)
ref_texts["Sad"] = (
    "Mrzí mě, že musím říct, že jsme utrpěli vážnou ránu v našem úsilí obnovit prosperitu a důvěru."
)
ref_texts["Angry"] = (
    "Obor astronomie je vtip! Jeho teorie jsou založeny na chybných pozorováních a zaujatých interpretacích!"
)
ref_texts["Surprised"] = (
    "Nemohu tomu uvěřit! Chcete mi říct, že jste v tomto rybníku objevili nový druh bakterie?"
)

In [ ]:
# Set up TPP for reference texts
ref_tpp = TppTtstool("cs-cz", tts_tool_bin=TTSTOOL_BIN_PATH, tts_tool_data=TTSTOOL_DATA_PATH)

# Unseen speaker
path = "Data/SPT-MGW.cs/SPT-MGW/spkr00001/wavs/spkr00001_Speaker001_1_0000021.wav"
s_ref = compute_style(path)

text = "Ano, jeho ctihodná velebnost je uvnitř, ale má u sebe jednoho či dva zbožné kazatele, a rovněž ranhojiče."
for k, v in ref_texts.items():

    wavs = STsynthesize(
        text,
        model,
        tpp,
        text_cleaner,
        s_ref,
        v,
        ref_tpp,
        sampler,
        diffusion_steps=10,
        embedding_scale=1.5,
        alpha=0.5,
        beta=0.9,  # make it more suitable for the text
        silence_beg=4800,
        silence_end=4800,
        device=DEVICE,
    )
    wav = np.concatenate(wavs)

    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

### Speech diversity

This section reproduces samples in [Section 7](https://styletts2.github.io/#var) of the demo page. 

`alpha` and `beta` determine the diversity of the synthesized speech. There are two extreme cases:
- If `alpha = 1` and `beta = 1`, the synthesized speech sounds the most dissimilar to the reference speaker, but it is also the most diverse (each time you synthesize a speech it will be totally different). 
- If `alpha = 0` and `beta = 0`, the synthesized speech sounds the most siimlar to the reference speaker, but it is deterministic (i.e., the sampled style is not used for speech synthesis). 


#### Default setting (`alpha = 0.3, beta=0.7`)
This setting uses 70% of the reference timbre and 30% of the reference prosody and use the diffusion model to sample them based on the text. 

In [ ]:
# unseen speaker
path = "Data/SPT-MGW.cs/SPT-MGW/spkr00001/wavs/spkr00001_Speaker001_1_0000021.wav"
ref_s = compute_style(path)

text = "Jak velká variabilita je v tomto projevu?"
for _ in range(5):
    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=1,
        alpha=0.3,
        beta=0.7,
        silence_beg=4800,
        silence_end=4800,
        device="cuda",
    )
    wav = np.concatenate(wavs)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### Less diverse setting (`alpha = 0.1, beta=0.3`)
This setting uses 90% of the reference timbre and 70% of the reference prosody. This makes it more similar to the reference speaker at cost of less diverse samples. 

In [ ]:
# unseen speaker
path = "Data/SPT-MGW.cs/SPT-MGW/spkr00001/wavs/spkr00001_Speaker001_1_0000021.wav"
ref_s = compute_style(path)

text = "Jak velká variabilita je v tomto projevu?"
for _ in range(5):
    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=1,
        alpha=0.1,
        beta=0.3,
        silence_beg=4800,
        silence_end=4800,
        device="cuda",
    )
    wav = np.concatenate(wavs)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### More diverse setting (`alpha = 0.5, beta=0.95`)
This setting uses 50% of the reference timbre and 5% of the reference prosody (so it uses 100% of the sampled prosody, which makes it more diverse), but this makes it more dissimilar to the reference speaker.  

In [ ]:
# unseen speaker
path = "Data/SPT-MGW.cs/SPT-MGW/spkr00001/wavs/spkr00001_Speaker001_1_0000021.wav"
ref_s = compute_style(path)

text = "Jak velká variabilita je v tomto projevu?"
for _ in range(5):
    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=1,
        alpha=0.5,
        beta=0.95,
        silence_beg=4800,
        silence_end=4800,
        device="cuda",
    )
    wav = np.concatenate(wavs)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### Extreme setting (`alpha = 1, beta=1`)
This setting uses 0% of the reference timbre and prosody and use the diffusion model to sample the entire style. This makes the speaker very dissimilar to the reference speaker. 

In [ ]:
# unseen speaker
path = "Data/SPT-MGW.cs/SPT-MGW/spkr00001/wavs/spkr00001_Speaker001_1_0000021.wav"
ref_s = compute_style(path)

text = "Jak velká variabilita je v tomto projevu?"
for _ in range(5):
    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=1,
        alpha=1,
        beta=1,
        silence_beg=4800,
        silence_end=4800,
        device="cuda",
    )
    wav = np.concatenate(wavs)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### No variation (`alpha = 0, beta=0`)
This setting uses 0% of the reference timbre and prosody and use the diffusion model to sample the entire style. This makes the speaker very similar to the reference speaker, but there is no variation. 

In [ ]:
# unseen speaker
path = "Data/SPT-MGW.cs/SPT-MGW/spkr00001/wavs/spkr00001_Speaker001_1_0000021.wav"
ref_s = compute_style(path)

text = "Jak velká variabilita je v tomto projevu?"
for _ in range(5):
    wavs = synthesize(
        text,
        model,
        tpp,
        text_cleaner,
        ref_s,
        sampler,
        diffusion_steps=10,
        embedding_scale=1,
        alpha=0,
        beta=0,
        silence_beg=4800,
        silence_end=4800,
        device="cuda",
    )
    wav = np.concatenate(wavs)
    display(ipd.Audio(wav, rate=24000, normalize=False))

### Extra fun!

Here we clone some of the authors' voice of the StyleTTS 2 papers with a few seconds of the recording in the wild. None of the voices is in the dataset and all authors agreed to have their voices cloned here.

In [ ]:
text = """ StyleTTS 2 is a text to speech model that leverages style diffusion and adversarial training with large speech language models to achieve human level text to speech synthesis. """

In [ ]:
reference_dicts = {}
reference_dicts["Yinghao"] = "Demo/reference_audio/Yinghao.wav"
reference_dicts["Gavin"] = "Demo/reference_audio/Gavin.wav"
reference_dicts["Vinay"] = "Demo/reference_audio/Vinay.wav"
reference_dicts["Nima"] = "Demo/reference_audio/Nima.wav"

In [ ]:
start = time.time()
noise = torch.randn(1, 1, 256).to(device)
for k, path in reference_dicts.items():
    ref_s = compute_style(path)

    wav = inference(text, ref_s, alpha=0.1, beta=0.5, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print("Speaker: " + k)
    import IPython.display as ipd

    print("Synthesized:")
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print("Reference:")
    display(ipd.Audio(path, rate=24000, normalize=False))